In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Read the uploaded Excel file
# Use openpyxl engine per instructions
try:
    df = pd.read_excel('CardioGoodFitness.xlsx', engine='openpyxl')
except Exception as e:
    # Fallback: sometimes sheet is not standard; try default
    df = pd.read_excel('CardioGoodFitness.xlsx')

# Basic structure
shape = df.shape
columns = df.columns.tolist()
dtypes = df.dtypes.astype(str)
missing = df.isna().sum()

# Clean column names (strip spaces if any)
df.columns = [c.strip() for c in df.columns]

# Convert to appropriate types if needed
# Ensure numeric columns are numeric
for col in ['Age','Education','Usage','Fitness','Income','Miles']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Descriptive statistics for numeric columns
num_cols = ['Age','Education','Usage','Fitness','Income','Miles']
desc = df[num_cols].describe().T

# Categorical summaries
cat_cols = ['Product','Gender','MaritalStatus']
cat_levels = {c: df[c].value_counts() for c in cat_cols}

# Product-level profile
prod_grp = df.groupby('Product').agg(
    n=('Product','size'),
    age_mean=('Age','mean'), age_median=('Age','median'),
    usage_mean=('Usage','mean'), fitness_mean=('Fitness','mean'),
    income_mean=('Income','mean'), miles_mean=('Miles','mean')
).reset_index()

# Gender by product (percentage)
gender_by_prod = (df.pivot_table(index='Product', columns='Gender', values='Age', aggfunc='count', fill_value=0))
gender_by_prod_pct = gender_by_prod.div(gender_by_prod.sum(axis=1), axis=0) * 100

# Marital status by product (percentage)
marital_by_prod = (df.pivot_table(index='Product', columns='MaritalStatus', values='Age', aggfunc='count', fill_value=0))
marital_by_prod_pct = marital_by_prod.div(marital_by_prod.sum(axis=1), axis=0) * 100

# Correlations
corr = df[num_cols].corr()

# Simple pairwise correlation highlights
corr_pairs = corr.unstack().sort_values(ascending=False)
# Exclude self-correlations
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
# Get top positive and negative unique pairs
# We'll take the first occurrence of each pair (A,B) and skip (B,A)
seen = set()
unique_pairs = []
for (a,b), v in corr_pairs.items():
    if (b,a) in seen or a==b:
        continue
    seen.add((a,b))
    unique_pairs.append(((a,b), v))

top_pos = [p for p in unique_pairs if p[1] > 0]
TopPos5 = top_pos[:5]

top_neg = [p for p in unique_pairs if p[1] < 0]
TopNeg5 = top_neg[-5:]  # since list is descending, negatives are at end; take last 5 (most negative)

# Visualizations directory
outdir = Path('eda_outputs')
outdir.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

# 1. Product distribution
plt.figure(figsize=(6,4))
ax = sns.countplot(x='Product', data=df, order=df['Product'].value_counts().index, palette='Set2')
plt.title('Customer Count by Product')
plt.xlabel('Product')
plt.ylabel('Count')
plt.tight_layout()
prod_dist_path = outdir / 'product_distribution.png'
plt.savefig(prod_dist_path)
plt.close()

# 2. Income by Product (boxplot)
plt.figure(figsize=(7,4))
ax = sns.boxplot(x='Product', y='Income', data=df, order=df['Product'].value_counts().index, palette='Set3')
plt.title('Income by Product')
plt.xlabel('Product')
plt.ylabel('Annual Income')
plt.tight_layout()
inc_box_path = outdir / 'income_by_product.png'
plt.savefig(inc_box_path)
plt.close()

# 3. Miles vs Usage scatter colored by Product
plt.figure(figsize=(6,4))
ax = sns.scatterplot(x='Usage', y='Miles', hue='Product', data=df, palette='Set1')
plt.title('Miles vs Usage by Product')
plt.xlabel('Weekly Usage (times/week)')
plt.ylabel('Miles Run')
plt.tight_layout()
scatter_path = outdir / 'miles_vs_usage.png'
plt.savefig(scatter_path)
plt.close()

# 4. Correlation heatmap
plt.figure(figsize=(6,5))
ax = sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
heatmap_path = outdir / 'correlation_heatmap.png'
plt.savefig(heatmap_path)
plt.close()

# 5. Age distribution histogram by Product
plt.figure(figsize=(7,4))
for prod in df['Product'].unique():
    sns.kdeplot(df.loc[df['Product']==prod,'Age'], label=prod, fill=True, alpha=0.3)
plt.title('Age Distribution by Product')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend(title='Product')
plt.tight_layout()
age_kde_path = outdir / 'age_distribution.png'
plt.savefig(age_kde_path)
plt.close()

# Summaries to dict for printing later
summary = {
    'shape': shape,
    'columns': columns,
    'dtypes': dtypes.to_dict(),
    'missing': missing.to_dict()
}

# Save key tables to CSV for traceability
prod_grp_path = outdir / 'product_profile.csv'
prod_grp.to_csv(prod_grp_path, index=False)

gender_by_prod_pct_path = outdir / 'gender_by_product_pct.csv'
gender_by_prod_pct.to_csv(gender_by_prod_pct_path)

marital_by_prod_pct_path = outdir / 'marital_by_product_pct.csv'
marital_by_prod_pct.to_csv(marital_by_prod_pct_path)

# Return textual summaries
result = {
    'summary': summary,
    'numeric_desc': desc.round(2).to_dict(),
    'product_profile_head': prod_grp.round(2).to_dict(orient='records'),
    'gender_by_product_pct': gender_by_prod_pct.round(1).to_dict(),
    'marital_by_product_pct': marital_by_prod_pct.round(1).to_dict(),
    'top_pos_corrs': [(a,b, round(v,2)) for (a,b), v in TopPos5],
    'top_neg_corrs': [(a,b, round(v,2)) for (a,b), v in TopNeg5],
    'plots': {
        'product_distribution': str(prod_dist_path),
        'income_by_product': str(inc_box_path),
        'miles_vs_usage': str(scatter_path),
        'correlation_heatmap': str(heatmap_path),
        'age_distribution': str(age_kde_path)
    }
}

result
